In [ ]:
#TODO
import sys
import json
import math
import sqlite3
from datetime import datetime
from kafka import KafkaConsumer

with open("config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

BOOTSTRAP_SERVER = config["kafka"]["bootstrap_server"]
TOKENS_TOPIC = config["kafka"]["tokens_topic"]
BASELINES_DB = config["baselines_db_path"]

# 2. Load precomputed token baselines into memory for O(1) rate lookups
conn = sqlite3.connect(BASELINES_DB)
cursor = conn.cursor()
cursor.execute("SELECT * FROM token_baselines")
baseline_lookup = {row[0]: list(row[1:]) for row in cursor.fetchall()}
conn.close()

print(f"Loaded baselines for {len(baseline_lookup)} tokens.")

# 3. Leaky Bucket Math
def update_level(current_level, arrivals, delta_t, baseline_hourly):
    """
    Computes time-decay drain based on elapsed seconds and checks spike threshold.
    """
    thresh_hold = max(7, 3 * math.sqrt(baseline_hourly))
    r = 2  # Drain multiplier
    
    drain = r * (baseline_hourly / 3600.0) * delta_t
    current_level = max(0.0, current_level - drain + arrivals)
    is_alert = (current_level >= thresh_hold)
    
    return current_level, is_alert, thresh_hold

# 4. Isolated in-memory state tracker for assigned partitions
# bucket_state: { token_str: {"level": float, "last_ts": int} }
bucket_state = {}

# 5. Kafka Consumer with Consumer Group for Automatic Partition Balancing
consumer = KafkaConsumer(
    TOKENS_TOPIC,
    bootstrap_servers=BOOTSTRAP_SERVER,
    group_id="leaky-bucket-evaluator-group",
    auto_offset_reset='latest',
    enable_auto_commit=True,
    key_deserializer=lambda k: k.decode('utf-8') if k else None,
    value_deserializer=lambda v: json.loads(v.decode('utf-8'))
)

print(f"Stage 2 Worker started. Subscribed to '{TOKENS_TOPIC}' under group 'leaky-bucket-evaluator-group'...")

try:
    for msg in consumer:
        token = msg.key
        if not token:
            continue
            
        ts = msg.value.get("ts", int(datetime.now().timestamp()))
        dt = datetime.fromtimestamp(ts)
        hour = dt.hour
        
        # Initialize state for token on first sight in this worker instance
        if token not in bucket_state:
            bucket_state[token] = {"level": 0.0, "last_ts": ts}
            
        state = bucket_state[token]
        delta_t = max(0.0, ts - state["last_ts"])
        
        # Look up baseline arrival rate for this specific hour (0-23)
        baseline_hourly = baseline_lookup.get(token, [0.01] * 24)[hour]
        
        # Update water level
        new_level, is_alert, thresh = update_level(
            current_level=state["level"],
            arrivals=1,
            delta_t=delta_t,
            baseline_hourly=baseline_hourly
        )
        
        state["level"] = new_level
        state["last_ts"] = ts
        
        time_str = dt.strftime("%H:%M:%S")
        print(f"[{time_str}] [Partition {msg.partition}] Token: '{token}' | Level: {new_level:5.2f} / {thresh:.1f} | Baseline(h{hour}): {baseline_hourly:.4f}")
        
        if is_alert:
            print(f"SPIKE ALERT! Token '{token}' exceeded threshold ({thresh:.1f}) with level {new_level:.2f} on Partition {msg.partition}!")

except KeyboardInterrupt:
    print("\nWorker stopped.")
finally:
    consumer.close()